# Neural Network reproduction

The dataset handling and neural-network workflow originate from `linphotonicslab/ML_Pipeline` (MIT License).  
This is a **modified reproduction notebook**, not a neural-network implementation claimed from scratch.

My changes in the reproduced version include current-device handling (CUDA/CPU), an additional training-feature scaling step, modified Optuna search/training settings, and rerunning/evaluating the model. The upstream pipeline and publication remain the methodological starting point.


In [ ]:
%reset -f
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

OUTPUT_TEST = True
EPOCHS = 50
BATCH_SIZE = 64
LOSS_FN = nn.MSELoss()


In [ ]:
X_train = pd.read_csv("../data/cleaned/training.csv")
y_train = pd.read_csv("../data/cleaned/training_labels.csv")
X_val = pd.read_csv("../data/cleaned/validation.csv")
y_val = pd.read_csv("../data/cleaned/validation_labels.csv")

for col in list(X_train.columns):
    if "[" in col or "]" in col:
        new_col = col.replace("[", "(").replace("]", ")")
        X_train = X_train.rename(columns={col: new_col})
        X_val = X_val.rename(columns={col: new_col})

X_train, X_verif, y_train, y_verif = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)
for obj in (X_train, y_train, X_verif, y_verif, X_val, y_val):
    obj.reset_index(drop=True, inplace=True)

# Current reproduced notebook applies a train-fitted scaler before NN training.
nn_scaler = StandardScaler()
X_train = pd.DataFrame(
    nn_scaler.fit_transform(X_train), columns=X_train.columns
)
X_verif = pd.DataFrame(
    nn_scaler.transform(X_verif), columns=X_verif.columns
)
X_val = pd.DataFrame(
    nn_scaler.transform(X_val), columns=X_val.columns
)


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features_dataframe, target_dataframe):
        self.features = features_dataframe
        self.target = target_dataframe

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features.iloc[idx].values, dtype=torch.float32)
        y = torch.tensor(self.target.iloc[idx].values, dtype=torch.float32)
        return x, y


def make_loader(X, y, shuffle=False):
    return DataLoader(
        CustomDataset(X, y),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


train_loader = make_loader(X_train, y_train, shuffle=True)
verif_loader = make_loader(X_verif, y_verif)
val_loader = make_loader(X_val, y_val)


In [ ]:
def define_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 6)
    layers = []
    in_features = X_train.shape[1]

    for i in range(n_layers):
        out_features = trial.suggest_int(f"n_units_l{i}", 4, 2048)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.LeakyReLU(0.01))
        p = trial.suggest_float(f"dropout_l{i}", 0.2, 0.5)
        layers.append(nn.Dropout(p))
        in_features = out_features

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)


class NeuralNetwork(nn.Module):
    def __init__(self, params):
        super().__init__()
        layers = []
        in_features = X_train.shape[1]

        for i in range(params["n_layers"]):
            out_features = params[f"n_units_l{i}"]
            layers.extend([
                nn.Linear(in_features, out_features),
                nn.LeakyReLU(0.01),
                nn.Dropout(params[f"dropout_l{i}"]),
            ])
            in_features = out_features

        layers.append(nn.Linear(in_features, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


In [ ]:
def evaluate_loader(model, loader):
    model.eval()
    true_values, predictions = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            pred = model(X).view(-1).cpu().numpy()
            true = y.view(-1).numpy()
            predictions.extend(pred)
            true_values.extend(true)

    true_values = np.asarray(true_values)
    predictions = np.asarray(predictions)
    rmse = mean_squared_error(true_values, predictions, squared=False)
    mape = mean_absolute_percentage_error(true_values, predictions) * 100
    return rmse, mape, true_values, predictions


def objective(trial):
    model = define_model(trial).to(device)

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["Adam", "RMSprop", "SGD"]
    )
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    for epoch in range(EPOCHS):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device).view(-1)
            optimizer.zero_grad()
            pred = model(X).view(-1)
            loss = LOSS_FN(pred, y)
            loss.backward()
            optimizer.step()

        verif_rmse, verif_mape, _, _ = evaluate_loader(model, verif_loader)
        score = verif_rmse + verif_mape
        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return score


In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.SuccessiveHalvingPruner(),
)
study.optimize(objective, n_trials=50)

params = study.best_trial.params
print("Best value:", study.best_trial.value)
print("Best parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")


In [ ]:
model = NeuralNetwork(params).to(device)
optimizer = getattr(optim, params["optimizer"])(
    model.parameters(), lr=params["lr"]
)

for epoch in range(EPOCHS):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device).view(-1)
        optimizer.zero_grad()
        pred = model(X).view(-1)
        loss = LOSS_FN(pred, y)
        loss.backward()
        optimizer.step()

val_rmse, val_mape, val_true, val_pred = evaluate_loader(model, val_loader)
print("Validation RMSE:", val_rmse)
print("Validation R²:", r2_score(val_true, val_pred))
print("Validation MAPE (%):", val_mape)


In [ ]:
if not OUTPUT_TEST:
    raise ValueError("Set OUTPUT_TEST=True to run the held-out test set.")

X_test = pd.read_csv("../data/cleaned/test.csv")
y_test = pd.read_csv("../data/cleaned/test_labels.csv")

for col in list(X_test.columns):
    if "[" in col or "]" in col:
        X_test = X_test.rename(
            columns={col: col.replace("[", "(").replace("]", ")")}
        )

X_test = pd.DataFrame(
    nn_scaler.transform(X_test), columns=X_test.columns
)
test_loader = make_loader(X_test, y_test)

test_rmse, test_mape, test_true, test_pred = evaluate_loader(model, test_loader)
print("Test RMSE:", test_rmse)
print("Test R²:", r2_score(test_true, test_pred))
print("Test MAPE (%):", test_mape)

from pathlib import Path
pred_dir = Path("../data/predictions/NN")
pred_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(test_pred).to_csv(pred_dir / "test_pred_nn.csv", index=False, header=False)
pd.DataFrame(test_true).to_csv(pred_dir / "test_true_nn.csv", index=False, header=False)
pd.DataFrame(X_test).to_csv(pred_dir / "test_input_nn.csv", index=False, header=False)
